# Benchmark sinh ảnh: bước, guidance, thời gian, VRAM, CLIP score

Bạn đo một pipeline `diffusers` như đo một service: chất lượng (CLIP score, và mắt bạn), thời gian mỗi ảnh, VRAM đỉnh,
trên một lưới cấu hình, rồi viết khuyến nghị cấu hình cho một sản phẩm cụ thể. Model chính là `stabilityai/sd-turbo`
(GPU 4GB hoặc CPU); model lớn hơn chỉ ở phần mở rộng trên PC 12GB.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`. Test kiểm chúng trên dữ liệu nhỏ, không tải model.
- Phần chạy thật nằm trong cell `explore` của bạn. Chạy GPU: mở notebook bằng kernel `ai-gpu` (`setup/gpu`), vì env gốc chỉ
  có torch CPU.
- Chấm: `uv run pytest tracks/ai-engineer/modules/18-multimodal/exercises/advanced/02-image-gen-benchmark -q`

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import statistics
import time
from typing import Callable

import numpy as np

## Ba hàm đo

1. `clip_score(image_emb, text_emb) -> np.ndarray`: hai mảng `(n, d)`, hàng $$i$$ là cặp (ảnh $$i$$, prompt của nó), **chưa**
   chuẩn hoá. Trả mảng `(n,)` với $$100\cdot\max(\cos(u_i, v_i), 0)$$ (Hessel et al., 2021). Hai mảng khác shape thì
   `ValueError`.
2. `time_call(fn, *, sync=None, warmup=1, repeats=3) -> float`: gọi `fn()` `warmup` lần không tính giờ, rồi `repeats` lần có
   tính giờ bằng `time.perf_counter`; nếu có `sync` thì gọi `sync()` **ngay trước khi bắt đầu và ngay trước khi dừng** đồng
   hồ mỗi lần (với CUDA là `torch.cuda.synchronize`, vì lệnh GPU chạy bất đồng bộ). Trả **trung vị** số giây. `repeats < 1`
   thì `ValueError`.
3. `pareto_front(rows, cost, quality) -> list[dict]`: các dòng **không bị trội**: dòng $$a$$ trội $$b$$ khi
   $$a_{\text{cost}} \le b_{\text{cost}}$$, $$a_{\text{quality}} \ge b_{\text{quality}}$$ và ít nhất một bất đẳng thức chặt.
   Trả theo `cost` tăng dần (hoà thì giữ thứ tự gốc); dòng trùng hệt cả hai giá trị thì giữ cả hai.

In [ ]:
def clip_score(image_emb: np.ndarray, text_emb: np.ndarray) -> np.ndarray:
    """100 * max(cos, 0) theo từng cặp hàng."""
    # TODO
    raise NotImplementedError


def time_call(fn: Callable[[], object], *, sync: Callable[[], None] | None = None, warmup: int = 1, repeats: int = 3) -> float:
    """Trung vị số giây của `repeats` lần gọi fn, sau `warmup` lần khởi động; sync trước khi bắt và dừng đồng hồ."""
    # TODO
    raise NotImplementedError


def pareto_front(rows: list[dict], cost: str, quality: str) -> list[dict]:
    """Các dòng không bị trội (cost thấp, quality cao), theo cost tăng dần."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: benchmark của bạn (xem README): nạp sd-turbo + CLIP, lưới cấu hình, đo, lưu ảnh và bảng, vẽ Pareto